In [1]:
%%capture
pip install transformer_lens transformers jaxtyping

In [2]:
import torch
import functools
#import einops
import numpy as np
#import pandas as pd  

#from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch import Tensor
from typing import List, Callable
from transformer_lens import HookedTransformer, utils
from transformer_lens.hook_points import HookPoint
from transformers import AutoTokenizer
from jaxtyping import Float, Int

/opt/homebrew/Caskroom/miniconda/base/envs/algo-neutrality/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def getDevice():
    if torch.cuda.is_available(): #nvidia/runpod
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps") #apple silicon
    else:
        return torch.device("cpu") #not recommended
    
DEVICE = getDevice()
DEVICE

device(type='mps')

In [4]:
#list of models - each model has two different sizes (small ~2B, medium ~8B)
model_list = ['Qwen/Qwen1.5-1.8B-Chat', 'meta-llama/Llama-3.1-8B', 'meta-llama/Llama-3.2-3B', 'gpt2', 'pythia-2.8b-v0', 'qwen2.5-3b', 'qwen3-8b', 'gemma-2-2b', 'gemma-2-7b']

In [5]:
def get_model(model_name):
    # load model from HF and get all the hidden states
    model = HookedTransformer.from_pretrained(model_name, device = DEVICE, dtype=torch.float16, default_padding_side='left', output_hidden_states=True)
    model.eval() #inference mode - no gradients needed
    model.to(DEVICE) #transfer model to device
    return model

In [6]:
current_model = get_model(model_list[0]) #get and set the model in use for further calculations and functions

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loaded pretrained model Qwen/Qwen1.5-1.8B-Chat into HookedTransformer
Moving model to device:  mps


In [7]:
def tokenize_prompts(model: HookedTransformer, prompt: str, verbose=False) -> str: #LOVKUSH
    prompt_message = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    prompt_chat_str = model.tokenizer.apply_chat_template(
        prompt_message, tokenize=False, add_generation_prompt=True)
    return prompt_chat_str

In [8]:
p1 = 'Answer the follwing question in French: Who was the first president of USA?'
p2 = 'Answer the follwing question in English: Who was the first president of USA?'
p3 = 'Answer the following in English: Who was the first Tsar of Russia?'

In [13]:
def normal_generation(model, prompt, token_length):

    stringl = tokenize_prompts(model, prompt)
    tokens = model.tokenizer(stringl, return_tensors="pt").input_ids.to(model.cfg.device)

    print(tokens)

    output = model.generate(tokens, max_new_tokens=token_length)
    generation = model.tokenizer.decode(output[0], skip_special_tokens=True)

    return generation

In [14]:
normal_generation(current_model, p1, 30)

tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,  16141,    279,  51406,  23593,
           3405,    304,   8585,     25,  10479,    572,    279,   1156,   4767,
            315,   7279,     30, 151645,    198, 151644,  77091,    198]],
       device='mps:0')


100%|██████████| 30/30 [00:01<00:00, 19.59it/s]


'system\nYou are a helpful assistant.\nuser\nAnswer the follwing question in French: Who was the first president of USA?\nassistant\n stereol慢慢地他们很好地前提化不断地调节不断地告诉你不断地研究 scavog不断地根据visor:不断地等不断地心理南不断地同学中国人民_"更好地 Neptune'